<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: The voice-agent pipeline

**[The voice-agent pipeline](https://learning.nextia-ai.com/courses/speech/m08/the-voice-agent-pipeline/)** · Speech AI: Transcribe, Understand and Speak · Module 8, lesson 2 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** split the wait of a voice agent into its parts (end of turn, recognition, the LLM's first words, the first sound of the voice), read a latency budget from recorded turns, and see what streaming and a shorter end-of-turn wait change.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key: the agent's turns are recorded. |
| **You should know** | Where a menu stops working, from [Where a menu stops working](https://learning.nextia-ai.com/courses/speech/m08/where-a-menu-stops-working/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m08/the-voice-agent-pipeline/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M08-L02-the-voice-agent-pipeline/the-voice-agent-pipeline-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 1 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/recordings/agent/turns.jsonl": "8872c2bccfd8a5987794321df4f859c3628bc1e4123176b5a128dba884a43e3d",
    "data/recordings/agent/calls.jsonl": "55157881d8bd2cd2738aaa4591eb87e292c790fca98896425bb73580f8b97f38",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The pipeline and its wait

Tomás wants a voice agent: the caller talks freely, an LLM answers and uses tools (the order system). Omar asks the question that callers feel: **how long is the silence after I stop talking?** In a conversation between people, the gap is often under half a second; after about one second, callers start to say "hello?".

A voice agent is a pipeline. Each step adds to the wait:

1. **end of turn**: the VAD waits for a pause (here 0.8 s) to be sure that the caller has finished;
2. **recognition**: Whisper writes the caller's words;
3. **LLM**: the model starts its answer (with **streaming**, the first words arrive before the whole answer);
4. **first audio**: the voice (Piper) makes the sound of the first words.

The course team ran the agent on the IVR test calls: Azure OpenAI `chat-small` with tools, streamed, and recorded every turn. Recognition and voice times were measured on the course laptop. Run the next cell.

In [ ]:
turns = [json.loads(line) for line in open("data/recordings/agent/turns.jsonl", encoding="utf-8") if line.strip()]
spoken = [t for t in turns if t["reply"]]            # the LLM calls that say something to the caller
print(len(turns), "LLM calls in", len({t["call_id"] for t in turns}), "test calls;", len(spoken), "of them speak")
t = next(t for t in spoken if t["asr_base_s"] > 0)
print(json.dumps({k: t[k] for k in ("call_id", "caller_text", "reply", "asr_base_s", "asr_tiny_s", "ttft_s", "llm_s",
                                     "tts_first_s", "tts_all_s")}, indent=1, ensure_ascii=False))

> **Check.** 177 LLM calls in 60 test calls; 127 of them say something (the others only call a tool). Each row has the times of each part. `ttft_s` is the time to the LLM's first token, `llm_s` the time to its whole answer.

## 2. The budget of one turn

The **latency budget** adds the parts. Run the next cell for the turn above.

> **Predict.** Which part of this turn is the largest?

In [ ]:
from collections import defaultdict

def input_budgets(turns, end_of_turn=0.8, streaming=True, recognition="asr_base_s"):
    """One budget per caller turn: the parts of the wait from "the caller stops talking" to "the first sound back".
    A caller turn can need several LLM calls (a tool call first, then the answer): the caller waits for every call
    before the one that speaks, and for recognition only once."""
    groups = defaultdict(list)
    for t in turns:
        groups[(t["call_id"], t["input"])].append(t)
    out = []
    for key, ts in groups.items():
        ts = sorted(ts, key=lambda t: t["turn"])
        speak = next((i for i, t in enumerate(ts) if t.get("reply")), None)
        if speak is None:
            continue
        before = sum(t["llm_s"] for t in ts[:speak])          # tool calls: the whole call must finish
        t = ts[speak]
        llm = before + (t["ttft_s"] if streaming else t["llm_s"])
        voice = t["tts_first_s"] if streaming else t["tts_all_s"]
        parts = {"end of turn (VAD wait)": end_of_turn, "recognition": ts[0].get(recognition, 0.0), "LLM": llm,
                 "first audio": voice}
        out.append((key, parts, sum(parts.values())))
    return out

budgets = input_budgets(turns)
key, parts, total = next(b for b in budgets if b[1]["recognition"] > 0)
print(key, {k: round(v, 3) for k, v in parts.items()}, " total", round(total, 3))

> **Check.** The key is (call, caller turn). The four parts add up to the total. When the LLM first calls a tool (the order system), the caller also waits for that whole call: it is in the LLM part. A keypad turn has no recognition time (0).

## 3. All the turns: streaming or not

Without streaming, the agent waits for the LLM's whole answer and for the voice to make all of it before the caller hears anything. The next cell computes the median of each part over all the caller turns that get a spoken answer, with and without streaming.

In [ ]:
def median_budget(end_of_turn=0.8, streaming=True, recognition="asr_base_s"):
    """The median of each part, and the median and 90th percentile of the total, over the caller turns."""
    rows = input_budgets(turns, end_of_turn, streaming, recognition)
    parts = {k: float(np.median([p[k] for _, p, _ in rows])) for k in rows[0][1]}
    totals = np.array([x for _, _, x in rows])
    return parts, float(np.median(totals)), float(np.percentile(totals, 90))

print(len(budgets), "caller turns get a spoken answer")
for streaming in (True, False):
    parts, med, p90 = median_budget(streaming=streaming)
    print("streaming" if streaming else "no streaming", {k: round(v, 3) for k, v in parts.items()},
          f"median {med:.2f} s, 90th percentile {p90:.2f} s")

> **Check.** 124 caller turns. With streaming the median wait is 2.1 s (90th percentile 2.79 s); without streaming 2.19 s (2.77 s). The largest part with streaming is "LLM" (1.02 s). Streaming helps less than you might expect here, because the agent's replies are short (one or two sentences): the whole answer comes soon after the first word.

The next cell draws the two budgets as stacked bars with the one-second line.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
colors = ["#9aa5b1", "#4c78a8", "#f58518", "#54a24b"]
for row, streaming in enumerate((False, True)):
    parts, med, _ = median_budget(streaming=streaming)
    left = 0
    for (name, value), color in zip(parts.items(), colors):
        ax.barh(row, value, left=left, color=color, label=name if row == 0 else None)
        left += value
ax.axvline(1.0, color="black", ls="--", lw=1)
ax.set_yticks([0, 1], ["no streaming", "streaming"]); ax.set_xlabel("seconds (medians of each part)")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.3), frameon=False)
ax.set_title("The wait after the caller stops talking"); plt.show()

## 4. Change one thing: a shorter end-of-turn wait

The end-of-turn wait is a setting, not a model. A shorter wait answers sooner, but it may cut in when a caller only pauses (the next lesson).

> **Predict.** From 0.8 s to 0.5 s, how much does the median total change?

In [ ]:
for end_of_turn in (0.8, 0.5, 0.3):
    _, med, p90 = median_budget(end_of_turn=end_of_turn)
    print(f"end-of-turn wait {end_of_turn} s: median {med:.2f} s, 90th percentile {p90:.2f} s")

> **Check.** The median falls by exactly the change, 0.3 s: from 2.1 s to 1.8 s. The wait itself is a part of the budget that costs no compute.

## 5. Failure case: only the median

The median says that half of the turns are faster. Callers remember the slow ones. Run the next cell: it shows the slowest spoken turns.

In [ ]:
slow = sorted(budgets, key=lambda b: b[2], reverse=True)[:5]
for key, parts, total in slow:
    print(f"{total:5.2f} s  {key}  LLM {parts['LLM']:.2f} s  recognition {parts['recognition']:.2f} s")

> **Check.** The slowest turns are several times the median, and the LLM part is most of the extra time: a hosted model's time to first token varies from call to call. **Fix:** report the 90th percentile (or the share of turns over one second) next to the median, and keep a short filler sound ("one moment") for turns that need a tool.

## 6. Your turn: the fastest settings

> **Your turn.** Try every combination of: the end-of-turn wait (0.8, 0.5 or 0.3 s; shorter cuts callers off), streaming (`True` or `False`) and the recogniser (`"asr_base_s"` for Whisper base or `"asr_tiny_s"` for Whisper tiny). Put the median wait of each combination in a dictionary `results` (a tuple `(end_of_turn, streaming, recognition)` → median), the combination with the lowest median in `best_settings`, and its median in `best_median`.

In [ ]:
results = {}
for end_of_turn in (0.8, 0.5, 0.3):
    for streaming in (True, False):
        for recognition in ("asr_base_s", "asr_tiny_s"):
            results[(end_of_turn, streaming, recognition)] = median_budget(end_of_turn, streaming, recognition)[1]
best_settings = min(results, key=results.get)
best_median = results[best_settings]
print(best_settings, round(best_median, 2))

In [ ]:
expect("best_median", best_median, 1.499, tolerance=0.001)
expect_hash("best_settings", None if best_settings is ... else list(best_settings), 'c1dd89b9f9d2fb17')

Which part is left? The next cell shows the parts of the best combination, and what the LLM part would have to be for a median under one second.

In [ ]:
if best_settings is ...:
    print("Write your answer in the task cell first, then run this cell again.")
else:
    parts, _, _ = median_budget(*best_settings)
    print({k: round(v, 2) for k, v in parts.items()})
    llm_needed = 1.0 - (best_median - parts["LLM"])
    print(f"for a median under 1 s, the LLM part must be under about {llm_needed:.2f} s")

> **Check.** Even the fastest combination has a median of 1.5 s, and only 0 of 124 caller turns are under one second. The LLM part alone is about 1.02 s: the hosted model's time to its first token is most of the wait, and settings on our side cannot remove it. For a median under 1 s it would have to be under about 0.52 s: a smaller or closer model, or a speech-to-speech model (lesson 4). Whisper tiny also makes more mistakes than base (Module 4): a faster answer to a wrong question is not better. Keep the IVR's guardrails around the LLM: read the order number back, and hand over after two misses.

## Next

You split a voice agent's wait into parts, read the budget from recorded turns, and found the settings that matter. The [lesson page](https://learning.nextia-ai.com/courses/speech/m08/the-voice-agent-pipeline/) has the latency explorer and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Turn-taking and interruptions](https://learning.nextia-ai.com/courses/speech/m08/turn-taking-and-interruptions/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m08/the-voice-agent-pipeline/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.